In [3]:
import ast
import re
import uuid
import logging
from typing import Optional, Tuple
import torch
import pandas as pd
from tqdm import tqdm

import psycopg2

from neo4j import GraphDatabase

from sentence_transformers import SentenceTransformer
import chromadb

C:\Users\Admin\Desktop\HK252\DATN\AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
def parse_spec(spec):
    if pd.isna(spec):
        return None

    try:
        spec_dict = ast.literal_eval(spec)
    except:
        return None

    if not isinstance(spec_dict, dict):
        return None

    return spec_dict


def extract_spec_key_samples(df):
    key_samples = {}

    for _, row in df.iterrows():
        spec_dict = parse_spec(row.get("spec"))

        if not spec_dict:
            continue

        for k, v in spec_dict.items():
            key = str(k).strip()

            if key not in key_samples:
                if v is not None and str(v).strip() != "":
                    key_samples[key] = str(v).strip()

    return key_samples


def build_global_spec_dictionary(csv_paths):
    global_samples = {}

    for name, path in csv_paths.items():
        print(f"Processing: {name}")

        df = pd.read_csv(path)

        samples = extract_spec_key_samples(df)

        for k, v in samples.items():
            if k not in global_samples:
                global_samples[k] = v

    return global_samples



csv_paths = {
    "laptop": "C:/Users/Admin/Desktop/HK252/DATN/scrape/data/laptop.csv",
    "man_hinh": "C:/Users/Admin/Desktop/HK252/DATN/scrape/data/man-hinh.csv",
    "may_in": "C:/Users/Admin/Desktop/HK252/DATN/scrape/data/may-in.csv",
    "micro": "C:/Users/Admin/Desktop/HK252/DATN/scrape/data/micro-thu-am.csv",
    "mobile": "C:/Users/Admin/Desktop/HK252/DATN/scrape/data/mobile.csv",
    "tablet": "C:/Users/Admin/Desktop/HK252/DATN/scrape/data/tablet.csv",
    "tivi": "C:/Users/Admin/Desktop/HK252/DATN/scrape/data/tivi.csv",
}

spec_dict = build_global_spec_dictionary(csv_paths)

print("\nTOTAL UNIQUE SPEC KEYS:", len(spec_dict))
print("\nSAMPLE OUTPUT:\n")

for i, (k, v) in enumerate(spec_dict.items()):
    print(f"{k} -> {v}")

Processing: laptop
Processing: man_hinh
Processing: may_in
Processing: micro
Processing: mobile
Processing: tablet
Processing: tivi

TOTAL UNIQUE SPEC KEYS: 64

SAMPLE OUTPUT:

Loại card đồ họa -> NVIDIA Geforce RTX 4050 6GB GDDR6. Intel UHD Graphics
Dung lượng RAM -> 16GB
Loại RAM -> DDR5 4800MHz
Số khe ram -> 2 khe (Hỗ trợ tối đa 32GB)
Ổ cứng -> 512GB SSD M.2 NVMe / 2 x M.2 NVMe
Kích thước màn hình -> 15.6 inches
Pin -> 4 cell 57 Wh , Pin liền
Hệ điều hành -> Windows 11 Home
Độ phân giải màn hình -> 1920 x 1080 pixels (FullHD)
Loại CPU -> Intel Core i5-12450H (2.0 GHz - 4.4 GHz, 18MB, 8 lõi / 12 luồng )
Cổng giao tiếp -> 1x HDMI. 3x USB 3.2. 1x Thunderbolt 4. Audio combo. LAN 1 Gb/s
Công nghệ màn hình -> Độ sáng 250nits. Độ phủ màu 45% NTSC. Acer ComfyView. Màn hình LCD TFT có đèn nền LED. Góc nhìn rộng lên đến 170 độ. Thiết kế siêu mỏng
Chip AI -> 572 TOPS
Treo tường -> 100 x 100 mm
Tính năng khác -> Cảm biến vân tay Touch ID, 720p FaceTime HD camera
Kích thước thực tế (bao gồm viề

In [5]:
SPEC_MAPPING = {
    # ===== COMPUTE / HARDWARE =====
    "Loại CPU": "cpu",
    "Chipset": "cpu",
    "Chip AI": "ai_chip",
    "Loại card đồ họa": "gpu",

    "Dung lượng RAM": "ram",
    "Loại RAM": "ram_type",
    "Số khe ram": "ram_slots",

    "Ổ cứng": "storage",
    "Bộ nhớ trong": "storage",

    # ===== DISPLAY =====
    "Kích thước màn hình": "screen_size",
    "Kích cỡ màn hình": "screen_size",
    "Kích thước thực tế (bao gồm viền)": "screen_size",

    "Độ phân giải màn hình": "resolution",
    "Độ phân giải": "resolution",

    "Tần số quét": "refresh_rate",
    "Thời gian phản hồi": "response_time",

    "Tấm nền": "panel_type",
    "Loại màn hình": "panel_type",
    "Công nghệ màn hình": "display_tech",
    "Tính năng màn hình": "display_features",

    "Tỉ lệ màn hình": "aspect_ratio",
    "Độ tương phản động": "contrast_ratio",

    # ===== CAMERA =====
    "Camera sau": "rear_camera",
    "Camera trước": "front_camera",

    # ===== BATTERY =====
    "Pin": "battery",
    "Dung lượng pin": "battery",
    "Lõi pin": "battery_type",

    # ===== OS =====
    "Hệ điều hành": "os",

    # ===== CONNECTIVITY / PORTS =====
    "Cổng giao tiếp": "ports",
    "Cổng kết nối": "ports",
    "Kết nối": "connectivity",
    "Cổng sạc vào": "charging_port",

    "Công nghệ NFC": "nfc",
    "Thẻ SIM": "sim",

    # ===== BUILD / PHYSICAL =====
    "Kích thước": "dimensions",
    "Trọng lượng": "weight",
    "Chất liệu": "material",
    "Treo tường": "mount_support",

    # ===== BRAND / META =====
    "Hãng sản xuất": "brand",
    "Thương hiệu": "brand",
    "Sản xuất tại": "made_in",
    "Năm ra mắt": "release_year",

    # ===== FEATURES =====
    "Tính năng khác": "features",
    "Tiện ích": "features",
    "Tiện ích nổi bật": "features",

    "Cảm biến": "sensors",

    # ===== AUDIO =====
    "Loại micro": "microphone_type",
    "Tỷ lệ lấy mẫu": "sample_rate",
    "Tỷ lệ tín hiệu trên tạp âm": "snr",
    "Tần số đáp ứng": "frequency_response",

    # ===== WIRELESS / RANGE =====
    "Phạm vi sử dụng": "range",

    # ===== PRINTER =====
    "Chức năng": "printer_functions",
    "Chất lượng in": "print_quality",
    "Thời gian in trang đầu": "first_page_time",
    "Tốc độ in": "print_speed",
    "Công suất in": "print_capacity",
    "Khay nạp giấy": "paper_tray",

    "Chất lượng Scan": "scan_quality",
    "Chất lượng Copy": "copy_quality",
    "Tốc độ Copy": "copy_speed",

    # ===== COMPATIBILITY =====
    "Tương thích": "compatibility",

    # ===== TV =====
    "Loại tivi": "tv_type",
    "Công nghệ hình ảnh": "image_tech",
    "Công nghệ âm thanh": "audio_tech",
}

NUMERIC_KEYS = {
    "ram",
    "storage",
    "screen_size",
    "refresh_rate",
    "response_time",
    "weight",
    "battery",
    "release_year",
}





def extract_price(price_str):
    if not price_str:
        return None

    price_str = price_str.replace(".", "")
    price_str = re.sub(r"[^\d]", "", price_str)

    return int(price_str) if price_str else None



UNIT_MAP = {
    "gb": "gb",
    "tb": "tb",
    "mb": "mb",

    "hz": "hz",

    "ms": "ms",

    "inch": "inch",
    "inches": "inch",

    "kg": "kg",
    "g": "g",

    "mah": "mah",
    "wh": "wh",

    "%": "%",
}
def normalize_unit(unit: str) -> str:
    return UNIT_MAP.get(unit.lower(), unit.lower())
def extract_number_unit(text: str) -> Tuple[Optional[float], Optional[str]]:
    if not text:
        return None, None

    text = str(text).lower().strip()

    text = text.replace('"', ' inch ')

    match = re.search(r"(\d+(?:\.\d+)?)\s*([a-zA-Z%]+)?", text)
    if not match:
        return None, None

    value = float(match.group(1))
    unit = match.group(2)

    if unit:
        unit = normalize_unit(unit)

    return value, unit



def parse_faq(faq_str):
    if pd.isna(faq_str) or faq_str.strip() == "":
        return []
    faqs = faq_str.split("### Câu hỏi:")
    qa_list = []
    for faq in faqs[1:]:
        parts = faq.split("### Câu trả lời:")
        if len(parts) == 2:
            question = parts[0].strip()
            answer = parts[1].strip()
            qa_list.append(f"{question}\n{answer}")
    return qa_list









logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)
logger = logging.getLogger(__name__)

device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_model = SentenceTransformer("dangvantuan/vietnamese-embedding", device=device)

def process_row(row, pg_conn, neo4j_driver, chroma_collection):
    product_id = str(uuid.uuid4())

    try:
        variant_list = ast.literal_eval(row.get("variant")) if row.get("variant") else []
    except Exception as e:
        logger.error(f"[VARIANT_PARSE_FAIL] product_id={product_id} error={e}")
        variant_list = []

    variants = []
    for v in variant_list:
        variants.append({
            "id": str(uuid.uuid4()),
            "color": v.get("color"),
            "price": extract_price(v.get("price")),
            "image": v.get("image")
        })

    try:
        spec_dict = ast.literal_eval(row.get("spec")) if row.get("spec") else {}
    except Exception as e:
        logger.error(f"[SPEC_PARSE_FAIL] product_id={product_id} error={e}")
        spec_dict = {}

    chunks = []

    try:
        desc = row.get("description")
        if pd.notna(desc) and str(desc).strip():
            chunks.append({
                "text": str(desc).strip(),
                "type": "description"
            })

        faq = row.get("faq")
        if pd.notna(faq) and str(faq).strip():
            faqs = parse_faq(faq)
            for f in faqs:
                chunks.append({
                    "text": f.strip(),
                    "type": "faq"
                })

    except Exception as e:
        logger.error(f"[CHUNK_BUILD_FAIL] product_id={product_id} error={e}")

    cur = pg_conn.cursor()

    try:
        cur.execute("""
            INSERT INTO product (id, name, category, brand, series, rating)
            VALUES (%s, %s, %s, %s, %s, %s)
        """, (
            product_id,
            row.get("name"),
            row.get("category"),
            row.get("brand"),
            row.get("series"),
            row.get("rating"),
        ))

        for v in variants:
            cur.execute("""
                INSERT INTO variant (id, product_id, color, price, image)
                VALUES (%s, %s, %s, %s, %s)
            """, (
                v["id"],
                product_id,
                v["color"],
                v["price"],
                v["image"],
            ))

        for raw_key, raw_value in spec_dict.items():
            raw_key = str(raw_key).strip()
            raw_value = str(raw_value).strip()

            canonical_key = SPEC_MAPPING.get(raw_key)
            if not canonical_key:
                continue

            value_num = None
            value_text = None
            unit = None

            if canonical_key in NUMERIC_KEYS:
                value_num, unit = extract_number_unit(raw_value)
                if value_num is None:
                    value_text = raw_value
            else:
                value_text = raw_value

            cur.execute("""
                INSERT INTO product_spec
                (product_id, spec_key, value_num, value_text, unit)
                VALUES (%s, %s, %s, %s, %s)
            """, (
                product_id,
                canonical_key,
                value_num,
                value_text,
                unit,
            ))

        pg_conn.commit()

    except Exception as e:
        pg_conn.rollback()
        logger.error(f"[POSTGRES_FAIL] product_id={product_id} error={e}")
        cur.close()
        raise RuntimeError(f"Postgres failed: {e}")

    cur.close()

    query = """
    MERGE (c:Category {name: $category})
    MERGE (b:Brand {name: $brand})
    MERGE (s:Series {name: $series})

    CREATE (p:Product {
        id: $product_id,
        name: $name,
        rating: $rating
    })

    MERGE (p)-[:IN_CATEGORY]->(c)
    MERGE (p)-[:OF_BRAND]->(b)
    MERGE (p)-[:IN_SERIES]->(s)

    WITH p
    UNWIND $variants AS v

    CREATE (var:Variant {
        id: v.id,
        color: v.color,
        price: v.price,
        image: v.image
    })

    MERGE (p)-[:HAS_VARIANT]->(var)
    """

    params = {
        "product_id": product_id,
        "name": row.get("name"),
        "category": row.get("category"),
        "brand": row.get("brand"),
        "series": row.get("series"),
        "rating": row.get("rating"),
        "variants": variants
    }

    try:
        with neo4j_driver.session() as session:
            session.execute_write(lambda tx: tx.run(query, params))

    except Exception as e:
        logger.error(f"[NEO4J_FAIL] product_id={product_id} error={e}")
        raise RuntimeError(f"Neo4j failed: {e}")

    try:
        docs = []
        metadatas = []
        ids = []

        for i, chunk in enumerate(chunks):
            text = chunk["text"].replace("\n\n---", "").strip()
            if not text:
                continue

            docs.append(text)

            metadatas.append({
                "product_id": product_id,
                "category": row.get("category"),
                "brand": row.get("brand"),
                "series": row.get("series"),
                "name": row.get("name"),
                "type": chunk["type"]
            })

            ids.append(f"{product_id}_{chunk['type']}_{i}")

        if docs:
            embeddings = embedding_model.encode(docs).tolist()

            chroma_collection.add(
                documents=docs,
                embeddings=embeddings,
                metadatas=metadatas,
                ids=ids
            )

    except Exception as e:
        logger.error(f"[CHROMA_FAIL] product_id={product_id} error={e}")


2026-04-19 01:51:09,806 | INFO | Load pretrained SentenceTransformer: dangvantuan/vietnamese-embedding
2026-04-19 01:51:10,538 | INFO | HTTP Request: HEAD https://huggingface.co/dangvantuan/vietnamese-embedding/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-04-19 01:51:10,553 | WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-04-19 01:51:10,614 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/dangvantuan/vietnamese-embedding/4ab46e46ba5902328ba0742e489e75f787932f2b/modules.json "HTTP/1.1 200 OK"
2026-04-19 01:51:10,888 | INFO | HTTP Request: HEAD https://huggingface.co/dangvantuan/vietnamese-embedding/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-04-19 01:51:10,944 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/dangvantuan/vietnamese-embedding/4ab46e46ba5902328ba0742e489e

In [6]:
pg_conn = psycopg2.connect(
    host="localhost",
    database="reldb",
    user="postgres",
    password="12345678"
)

neo4j_driver = GraphDatabase.driver(
    "neo4j://127.0.0.1:7687",
    auth=("neo4j", "12345678")
)

chroma_client = chromadb.PersistentClient(path="./vectordb")
collection_name = "products"
try:
    chroma_collection = chroma_client.get_collection(collection_name)
except:
    chroma_collection = chroma_client.create_collection(collection_name)

print("Connected to PostgreSQL & Neo4j & Chroma")





df_laptop = pd.read_csv("C:/Users/Admin/Desktop/HK252/DATN/scrape/data/laptop.csv")
df_man_hinh = pd.read_csv("C:/Users/Admin/Desktop/HK252/DATN/scrape/data/man-hinh.csv")
df_may_in = pd.read_csv("C:/Users/Admin/Desktop/HK252/DATN/scrape/data/may-in.csv")
df_micro = pd.read_csv("C:/Users/Admin/Desktop/HK252/DATN/scrape/data/micro-thu-am.csv")
df_mobile = pd.read_csv("C:/Users/Admin/Desktop/HK252/DATN/scrape/data/mobile.csv")
df_tablet = pd.read_csv("C:/Users/Admin/Desktop/HK252/DATN/scrape/data/tablet.csv")
df_tivi = pd.read_csv("C:/Users/Admin/Desktop/HK252/DATN/scrape/data/tivi.csv")
df = pd.concat([df_laptop, df_man_hinh, df_may_in, df_micro, df_mobile, df_tablet, df_tivi], ignore_index=True)

print("Loaded combined dataframe")




for idx, row in tqdm(df.iterrows(), total=len(df)):
    process_row(row, pg_conn, neo4j_driver, chroma_collection)

pg_conn.close()
neo4j_driver.close()

print("\nDONE - all connections closed")

Connected to PostgreSQL & Neo4j & Chroma
Loaded combined dataframe


Batches: 100%|████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 63.53it/s]

Batches: 100%|████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 67.27it/s]

Batches: 100%|████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 46.47it/s]

Batches: 100%|████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 63.88it/s]

Batches: 100%|████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 53.22it/s]

Batches: 100%|████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 69.34it/s]

Batches: 100%|████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 66.20it/s]

Batches: 100%|████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 59.2


DONE - all connections closed
